# 02 — Data Cleaning & Feature Engineering
**Input:** `data/raw/` — 9 CSVs originales  
**Output:** `exports/` — tabla maestra + tablas pre-agregadas para Power BI y SQL

Flujo:
1. Limpiar cada tabla individualmente
2. Construir tabla maestra (`orders_features.csv`) con todos los JOINs
3. Calcular métricas (revenue, delivery_days, is_late)
4. Generar tablas pre-agregadas por pregunta de negocio
5. Exportar todo con `encoding='utf-8-sig'` para Power BI

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

RAW     = '../data/raw/'
EXPORTS = '../exports/'
os.makedirs(EXPORTS, exist_ok=True)

print('Directorios listos.')

---
## 1. Carga de datos raw

In [ ]:
orders       = pd.read_csv(RAW + 'olist_orders_dataset.csv')
order_items  = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
payments     = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
reviews      = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')
customers    = pd.read_csv(RAW + 'olist_customers_dataset.csv')
products     = pd.read_csv(RAW + 'olist_products_dataset.csv')
sellers      = pd.read_csv(RAW + 'olist_sellers_dataset.csv')
geolocation  = pd.read_csv(RAW + 'olist_geolocation_dataset.csv')
translations = pd.read_csv(RAW + 'product_category_name_translation.csv')

print('Raw data cargada.')

---
## 2. Limpieza por tabla

In [ ]:
# --- orders ---
ts_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in ts_cols:
    orders[col] = pd.to_datetime(orders[col])

# Filtrar solo delivered para análisis principal
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print(f'orders total:     {len(orders):,}')
print(f'orders delivered: {len(orders_delivered):,} ({len(orders_delivered)/len(orders):.1%}')

In [ ]:
# --- order_items ---
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

# Verificar precios extremos
print(f'Items con price < 1: {(order_items["price"] < 1).sum()}')
print(order_items[order_items['price'] < 1][['order_id','price','freight_value']].head())

In [ ]:
# --- payments ---
# Eliminar not_defined e installments=0
before = len(payments)
payments = payments[payments['payment_type'] != 'not_defined']
payments = payments[payments['payment_installments'] > 0]
print(f'Payments eliminados: {before - len(payments)} filas')

# Agregar a nivel orden (puede haber múltiples métodos de pago por orden)
payments_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', lambda x: x.mode()[0])  # tipo más usado
).reset_index()

print(f'Payments agregados a nivel orden: {len(payments_agg):,}')

In [ ]:
# --- reviews ---
# Los nulls en comment_message y comment_title son válidos (campos opcionales)
# Agregar a nivel orden (quedarse con el review más reciente si hay duplicados)
reviews_clean = reviews.sort_values('review_creation_date').drop_duplicates(
    subset='order_id', keep='last'
)[['order_id', 'review_score']].copy()

dups = len(reviews) - len(reviews_clean)
print(f'Reviews duplicadas eliminadas: {dups}')
print(f'Reviews limpias: {len(reviews_clean):,}')

In [ ]:
# --- customers ---
# Renombrar para mayor claridad en joins
customers_clean = customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']].copy()
print(f'Customers: {len(customers_clean):,}')

In [ ]:
# --- products ---
# Corregir typos en nombres de columnas
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

# Agregar traducción al inglés
# Primero completar las 2 categorías sin traducción
extra_translations = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['pc_gamer', 'kitchen_portables_and_food_preparers']
})
translations_full = pd.concat([translations, extra_translations], ignore_index=True)

products_clean = products.merge(translations_full, on='product_category_name', how='left')
products_clean['product_category_name_english'] = products_clean['product_category_name_english'].fillna('unknown')

print(f'Products con categoría: {products_clean["product_category_name_english"].notna().sum():,}')
print(f'Products unknown: {(products_clean["product_category_name_english"] == "unknown").sum()}')

In [ ]:
# --- geolocation ---
# Deduplicar: una coordenada por zip code (mediana)
geo_clean = geolocation.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'median'),
    lng=('geolocation_lng', 'median')
).reset_index()

print(f'Geolocation: {len(geolocation):,} -> {len(geo_clean):,} filas')

---
## 3. Tabla maestra — orders_features

In [ ]:
# Base: orden x item (granularidad = 1 fila por item)
df = orders_delivered.merge(order_items, on='order_id', how='inner')
assert len(df) >= len(order_items[order_items['order_id'].isin(orders_delivered['order_id'])]), \
    "JOIN orders x order_items perdió filas inesperadamente"
print(f'Tras JOIN orders x items: {len(df):,}')

In [ ]:
# + customers
before = len(df)
df = df.merge(customers_clean, on='customer_id', how='left')
assert len(df) == before, f"JOIN customers multiplicó filas: {before} -> {len(df)}"

# + products (solo columnas útiles)
before = len(df)
df = df.merge(
    products_clean[['product_id', 'product_category_name_english',
                     'product_weight_g', 'product_length_cm',
                     'product_height_cm', 'product_width_cm']],
    on='product_id', how='left'
)
assert len(df) == before, f"JOIN products multiplicó filas: {before} -> {len(df)}"

# + sellers
before = len(df)
df = df.merge(
    sellers.rename(columns={'seller_state': 'seller_state', 'seller_city': 'seller_city'}),
    on='seller_id', how='left'
)
assert len(df) == before, f"JOIN sellers multiplicó filas: {before} -> {len(df)}"

# + payments (agregados a nivel orden)
before = len(df)
df = df.merge(payments_agg, on='order_id', how='left')
assert len(df) == before, f"JOIN payments multiplicó filas: {before} -> {len(df)}"

# + reviews
before = len(df)
df = df.merge(reviews_clean, on='order_id', how='left')
assert len(df) == before, f"JOIN reviews multiplicó filas: {before} -> {len(df)}"

print(f'Tabla maestra tras todos los JOINs: {df.shape}')

---
## 4. Feature Engineering

In [ ]:
# Revenue y métricas de delivery — calculadas en Python, no en DAX
df['revenue']       = df['price']  # precio del producto
df['total_cost']    = df['price'] + df['freight_value']  # costo total al cliente

df['delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

df['estimated_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.days

df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)

df['days_early_late'] = df['estimated_days'] - df['delivery_days']  # positivo = llegó antes

print('Métricas calculadas.')
print(df[['revenue', 'delivery_days', 'estimated_days', 'is_late', 'days_early_late']].describe())

In [ ]:
# Columnas de fecha para Power BI (ordenamiento y slicers)
df['purchase_year']       = df['order_purchase_timestamp'].dt.year
df['purchase_month']      = df['order_purchase_timestamp'].dt.month          # numérico 1-12
df['purchase_month_name'] = df['order_purchase_timestamp'].dt.strftime('%b') # Jan, Feb...
df['purchase_quarter']    = df['order_purchase_timestamp'].dt.quarter
df['purchase_day_of_week']= df['order_purchase_timestamp'].dt.dayofweek      # 0=lunes
df['purchase_year_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str) # 2017-01

# Columnas de ordenamiento para Power BI
df['month_order'] = df['purchase_month']  # ya es numérico, Power BI puede ordenar por esto

day_name_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday',
                3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df['day_of_week_name']  = df['purchase_day_of_week'].map(day_name_map)
df['day_of_week_order'] = df['purchase_day_of_week']

print('Columnas de fecha y ordenamiento agregadas.')
print(df[['purchase_year', 'purchase_month', 'purchase_month_name',
          'purchase_quarter', 'purchase_year_month']].head(3))

In [ ]:
# IDs como string (evitar que Power BI los trate como números)
id_cols = ['order_id', 'customer_id', 'customer_unique_id', 'product_id', 'seller_id']
for col in id_cols:
    df[col] = df[col].astype(str)

print('dtypes finales de columnas clave:')
print(df[id_cols + ['revenue', 'delivery_days', 'is_late', 'review_score']].dtypes)

---
## 5. Tablas pre-agregadas

In [ ]:
# agg_monthly — tendencia de ventas mes a mes
agg_monthly = df.groupby(['purchase_year_month', 'purchase_year', 'purchase_month',
                           'purchase_month_name', 'month_order']).agg(
    revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique'),
    items=('order_item_id', 'count'),
    avg_ticket=('revenue', 'mean')
).reset_index().sort_values('purchase_year_month')

print(f'agg_monthly: {agg_monthly.shape}')
agg_monthly.head(3)

In [ ]:
# agg_category — revenue por categoría de producto
agg_category = df.groupby('product_category_name_english').agg(
    revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique'),
    items=('order_item_id', 'count'),
    avg_price=('price', 'mean'),
    avg_review=('review_score', 'mean')
).reset_index().sort_values('revenue', ascending=False)

print(f'agg_category: {agg_category.shape}')
agg_category.head(5)

In [ ]:
# agg_state — revenue y clientes por estado
agg_state = df.groupby('customer_state').agg(
    revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique'),
    customers=('customer_unique_id', 'nunique'),
    avg_delivery_days=('delivery_days', 'mean'),
    pct_late=('is_late', 'mean')
).reset_index().sort_values('revenue', ascending=False)

agg_state['pct_late'] = agg_state['pct_late'].round(4)  # 4 decimales para Power BI

print(f'agg_state: {agg_state.shape}')
agg_state.head(5)

In [ ]:
# agg_delivery — performance de entrega por estado del vendedor
agg_delivery = df.groupby('seller_state').agg(
    orders=('order_id', 'nunique'),
    avg_delivery_days=('delivery_days', 'mean'),
    avg_estimated_days=('estimated_days', 'mean'),
    pct_late=('is_late', 'mean'),
    late_orders=('is_late', 'sum')
).reset_index().sort_values('pct_late', ascending=False)

agg_delivery['pct_late'] = agg_delivery['pct_late'].round(4)

print(f'agg_delivery: {agg_delivery.shape}')
agg_delivery.head(5)

In [ ]:
# agg_sellers — top sellers por revenue
agg_sellers = df.groupby(['seller_id', 'seller_state', 'seller_city']).agg(
    revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique'),
    items=('order_item_id', 'count'),
    avg_price=('price', 'mean'),
    avg_review=('review_score', 'mean')
).reset_index().sort_values('revenue', ascending=False)

print(f'agg_sellers: {agg_sellers.shape}')
agg_sellers.head(5)

In [ ]:
# agg_payment — métodos de pago
agg_payment = df.drop_duplicates(subset='order_id').groupby('payment_type').agg(
    orders=('order_id', 'nunique'),
    revenue=('payment_value', 'sum'),
    avg_installments=('payment_installments', 'mean')
).reset_index().sort_values('orders', ascending=False)

print(f'agg_payment: {agg_payment.shape}')
agg_payment

---
## 6. Sanity checks finales

In [ ]:
print('=== SANITY CHECKS ===')

# Revenue total
rev_features = df['revenue'].sum()
rev_raw      = order_items[order_items['order_id'].isin(orders_delivered['order_id'])]['price'].sum()
print(f'Revenue features: R$ {rev_features:,.0f}')
print(f'Revenue raw (solo delivered): R$ {rev_raw:,.0f}')
assert abs(rev_features - rev_raw) < 1, f'Revenue no coincide: {rev_features} vs {rev_raw}'
print('OK — Revenue coincide')

# Nulls en columnas clave
key_cols = ['order_id', 'customer_unique_id', 'product_category_name_english',
             'seller_id', 'revenue', 'delivery_days']
nulls = df[key_cols].isnull().sum()
print(f'\nNulls en columnas clave:')
print(nulls)

# Filas totales
print(f'\nTabla maestra: {len(df):,} filas x {len(df.columns)} columnas')
print(f'Órdenes únicas: {df["order_id"].nunique():,}')
print(f'Clientes únicos: {df["customer_unique_id"].nunique():,}')
print(f'Sellers únicos: {df["seller_id"].nunique():,}')

---
## 7. Export — utf-8-sig para Power BI

In [ ]:
exports = {
    'orders_features.csv': df,
    'agg_monthly.csv':     agg_monthly,
    'agg_category.csv':    agg_category,
    'agg_state.csv':       agg_state,
    'agg_delivery.csv':    agg_delivery,
    'agg_sellers.csv':     agg_sellers,
    'agg_payment.csv':     agg_payment,
}

for filename, dataframe in exports.items():
    path = EXPORTS + filename
    dataframe.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'Exportado: {filename:35s} {str(dataframe.shape):15s} -> {path}')

print('\nTodos los archivos exportados correctamente.')